# DecisionScope CA1 Agent Demo

This notebook demonstrates the DecisionScope CA1 AI Agent. The agent evaluates proposed policy changes by gathering supporting evidence, simulating stakeholder impact, identifying risks, and recommending a final decision.

In [ ]:
// Setup the environment and agent
const path = require('path');
const dotenvPath = path.resolve(__dirname, '../apps/api/node_modules/dotenv');
try { require(dotenvPath).config({ path: path.resolve(__dirname, '../apps/api/.env') }); } catch(e) {}
const { createLLMProvider } = require('../apps/api/src/infrastructure/llm/createLLMProvider');
const { createPlannerNode, createEvidenceToolNode, createSimulationToolNode, createCritiqueToolNode, createFinalJudgeNode } = require('../apps/api/src/agents/decisionAgent');
const { createInputNode } = require('../apps/api/src/agents/index');
const { createDecisionGraph } = require('../apps/api/src/orchestration/decisionGraph');
const { createConversationMemory, buildMemoryContext, updateMemory } = require('../apps/api/src/agents/memory');

const llmProvider = createLLMProvider();
const agents = {
  inputNode: createInputNode(llmProvider),
  plannerNode: createPlannerNode(llmProvider),
  evidenceToolNode: createEvidenceToolNode(llmProvider, null),
  simulationToolNode: createSimulationToolNode(llmProvider),
  critiqueToolNode: createCritiqueToolNode(llmProvider),
  finalJudge: createFinalJudgeNode(llmProvider),
};
const decisionGraph = createDecisionGraph(agents);

async function executeGraph(decisionGraph, initialState) {
  const stream = await decisionGraph.stream(initialState);
  let finalState = {};
  for await (const chunk of stream) {
    const nodeName = Object.keys(chunk)[0];
    const stateUpdate = chunk[nodeName];
    finalState = { ...finalState, ...stateUpdate };
    if (stateUpdate.agentTrace && stateUpdate.agentTrace.events) {
      const events = stateUpdate.agentTrace.events;
      const event = events[events.length - 1];
      if (event) {
        console.log(`\nSTEP ${event.step} — ${event.type.toUpperCase()}`);
        if (event.type === 'tool_call') { console.log(`Tool: ${event.tool}\nInput: ${event.inputSummary}`); }
        else if (event.type === 'tool_result') { console.log(`Result: ${event.resultSummary}`); }
        else { console.log(`${event.message}`); }
      }
    }
  }
  if (finalState.finalRecommendation) {
    console.log(`\nSTEP ${finalState.agentTrace?.step + 1 || '?'} — FINAL\nRecommendation:\n${finalState.finalRecommendation}`);
  }
  return finalState;
}

## EXAMPLE 1 — Normal Policy Analysis

Goal: "Increase the minimum attendance requirement from 75% to 85%."

In [ ]:
const example1Id = 'demo-ex1';
createConversationMemory(example1Id);
const context1 = buildMemoryContext(example1Id);

const state1 = {
  decisionId: example1Id,
  conversationId: example1Id,
  policyContext: "University Attendance Policy",
  proposedAction: "Increase the minimum attendance requirement from 75% to 85%",
  memory: context1
};

console.log(`USER GOAL:\n${state1.proposedAction}\n`);
await executeGraph(decisionGraph, state1);

## EXAMPLE 2 — Ambiguous Input

Goal: "Make the attendance policy stricter."

In [ ]:
const example2Id = 'demo-ex2';
createConversationMemory(example2Id);
const context2 = buildMemoryContext(example2Id);

const state2 = {
  decisionId: example2Id,
  conversationId: example2Id,
  policyContext: "University Attendance Policy",
  proposedAction: "Make the attendance policy stricter",
  memory: context2,
  requiresClarification: true // Simulating input validation failure
};

console.log(`USER GOAL:\n${state2.proposedAction}\n`);
await executeGraph(decisionGraph, state2);

## EXAMPLE 3 — Memory Demonstration

TURN 1: "Analyze an attendance penalty increase from 500 INR to 1000 INR."
TURN 2: "What if we change the proposed penalty to 750 INR?"

In [ ]:
const example3Id = 'demo-ex3';
createConversationMemory(example3Id);

console.log("--- TURN 1 ---");
const context3_1 = buildMemoryContext(example3Id);
const state3_1 = {
  decisionId: example3Id,
  conversationId: example3Id,
  policyContext: "Attendance Penalty Policy",
  proposedAction: "Analyze an attendance penalty increase from 500 INR to 1000 INR",
  memory: context3_1
};

console.log(`USER GOAL:\n${state3_1.proposedAction}\n`);
const final3_1 = await executeGraph(decisionGraph, state3_1);

// Save memory
updateMemory(example3Id, {
  previousPolicy: final3_1.policyContext,
  previousProposedAction: final3_1.proposedAction,
  lastRecommendation: final3_1.finalRecommendation
});

console.log("\n--- TURN 2 ---");
const context3_2 = buildMemoryContext(example3Id);
const state3_2 = {
  decisionId: example3Id,
  conversationId: example3Id,
  policyContext: "Attendance Penalty Policy",
  proposedAction: "What if we change the proposed penalty to 750 INR?",
  memory: context3_2
};

console.log(`USER GOAL:\n${state3_2.proposedAction}\n`);
console.log(context3_2);
await executeGraph(decisionGraph, state3_2);